# Alpamayo-v1.5 Phase 3a — Gradient-guided VLM-weight BFA

Targets `vlm.model.layers.*` (Qwen3-VL-8B text decoder) Linear weights in BF16.
Ranks bit-flip coordinates by `∂loss/∂vlm_weight · Δw` from a single activation-checkpointed VLM backward, then re-prefills per trial to measure the actual post-flip FM loss.

**Why re-prefill is mandatory here.** Unlike Phase 1 (expert-weight flips), a VLM weight flip is *silent* against a cached K/V — the cache was computed with the clean weight and is never reread by the expert forward. The only way the flipped weight affects the loss is to rebuild the cache with the flipped weight in place, which costs ~600 ms VLM prefill per trial. The trial budget below is sized accordingly (16 bits × 50 modules × 5 coords ≈ 4k trials → ~45 min/scene on one H20).

**Why activation checkpointing.** The single VLM backward used to compute clean grads costs ~25–40 GB of activation memory at S≈2000 prefill tokens. Wrapping each `vlm.model.layers[i].forward` in `torch.utils.checkpoint` keeps peak memory under ~80 GB on a 96 GB H20. See `_checkpoint_vlm_layers` in `bfa_utils.py`.

**Shared-GPU note.** All CUDA placement and the BFA utility calls take `DEVICE` explicitly so this notebook never silently lands on `cuda:0`. Set `GPU_ID` below to your allocated GPU before running.

Output: `results/bfa_vlm_grad_<scene>.parquet`. Replay top-N through `bfa_cascade_ade.ipynb` for the actual ADE/FDE metric.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo1_5 import helper

import bfa_utils as bfa

In [ ]:
# Pin to a specific GPU. Edit GPU_ID per session — never rely on cuda:0
# defaults on the shared 8×H20 box.
GPU_ID = int(os.environ.get("ALPAMAYO_GPU", "0"))
DEVICE = torch.device(f"cuda:{GPU_ID}")
print(f"Using device: {DEVICE} (visible CUDA devices: {torch.cuda.device_count()})")

In [ ]:
model = Alpamayo1_5.from_pretrained("nvidia/Alpamayo-1.5-10B", dtype=torch.bfloat16).to(DEVICE)
model.eval()
# Need requires_grad on VLM weights for the bit-grad backward.
for p in model.parameters():
    p.requires_grad_(True)

processor = helper.get_processor(model.tokenizer)

clip_ids = pd.read_parquet("clip_ids.parquet")["clip_id"].tolist()
SCENE_IDX = 774  # same default as inference.ipynb / bfa_demo.ipynb
clip_id = clip_ids[SCENE_IDX]
data = load_physical_aiavdataset(clip_id)

messages = helper.create_message(
    data["image_frames"].flatten(0, 1),
    camera_indices=data["camera_indices"],
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
model_inputs = helper.to_device(
    {
        "tokenized_data": inputs,
        "ego_history_xyz": data["ego_history_xyz"],
        "ego_history_rot": data["ego_history_rot"],
    },
    DEVICE,
)

In [ ]:
# GT action and the clean (no-grad) FMContext used by the per-trial measurement.
# Note: this is a SEPARATE context from the with-grad one inside compute_clean_grads_vlm.
gt_action = model.action_space.traj_to_action(
    traj_history_xyz=data["ego_history_xyz"].to(DEVICE).to(torch.float32),
    traj_history_rot=data["ego_history_rot"].to(DEVICE).to(torch.float32),
    traj_future_xyz=data["ego_future_xyz"].to(DEVICE).to(torch.float32),
    traj_future_rot=data["ego_future_rot"].to(DEVICE).to(torch.float32),
)
gt_action = gt_action.view(-1, *model.action_space.get_action_space_dims())
print("gt_action:", gt_action.shape, gt_action.dtype, gt_action.device)

with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
    ctx = bfa.build_fm_context(model, model_inputs, gt_action)

# Fix noise so every trial uses the SAME x_t — only the weight flip varies.
with torch.cuda.device(DEVICE):
    torch.cuda.manual_seed(42)
fixed_noise = torch.randn_like(ctx.gt_action)
T_VAL = 0.5

def loss_fn():
    with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
        return bfa.fm_one_step_loss(model, ctx, t_val=T_VAL, noise=fixed_noise)

baseline = loss_fn().item()
print("baseline FM loss:", baseline)

In [ ]:
# Collect all VLM-text-decoder Linears, then deterministically subsample.
# Full target set is ~250 modules; the ~45-min budget allows ~50 modules at
# 16 bits × 5 coords × 0.65 s/trial.
all_targets = bfa.collect_target_linears_vlm(model)
print(f"total VLM target Linears: {len(all_targets)}")

MODULE_K = 50
rng = np.random.default_rng(0)
names_sorted = sorted(all_targets.keys())
subsample_idx = rng.choice(len(names_sorted), size=min(MODULE_K, len(names_sorted)), replace=False)
targets = {names_sorted[i]: all_targets[names_sorted[i]] for i in sorted(subsample_idx)}
print(f"subsampled targets: {len(targets)}")
print("first 5:", list(targets.keys())[:5])

# Single VLM backward — heavy, ~5-8 s on H20 with checkpointing.
t0 = time.time()
torch.cuda.reset_peak_memory_stats(DEVICE)
grads = bfa.compute_clean_grads_vlm(
    model, targets, model_inputs, gt_action,
    device=DEVICE,
    t_val=T_VAL, noise=fixed_noise,
)
peak_gb = torch.cuda.max_memory_allocated(DEVICE) / 1e9
print(f"VLM backward done in {time.time()-t0:.1f}s; peak mem on {DEVICE}: {peak_gb:.1f} GB")
print(f"example grad norm: {list(grads.values())[0].norm().item():.4f}")

## Smoke tests

Run these before committing to the ~45-min main loop. (a)–(d) below; (c) and (d) deliberately exercise the re-prefill path so failures here surface before the sweep.

In [ ]:
# (a) Backward reached the VLM and produced non-zero grads on a deep layer.
any_layer = next(iter(targets.values()))
g = grads[next(iter(targets.keys()))]
assert g.shape == any_layer.weight.shape, (g.shape, any_layer.weight.shape)
assert (g != 0).any().item(), "grad is identically zero — autograd graph likely broken"
print(f"backward reaches VLM: OK ({(g != 0).float().mean().item():.3f} fraction of coords nonzero)")

In [ ]:
# (b) Memory bound: peak during clean-grads must be safely under 90 GB on H20.
assert peak_gb < 90, f"peak {peak_gb:.1f} GB exceeded budget; checkpointing may not be applied"
print(f"memory bound: OK ({peak_gb:.1f} GB < 90 GB on {DEVICE})")

In [ ]:
# (c) Mantissa-LSB sanity: pick a VLM Linear, flip bit 0 at a random coord via
# the re-prefill path; expect small finite Δloss; restore returns to baseline.
name = list(targets.keys())[0]
mod = targets[name]
rng_smoke = np.random.default_rng(1)
rand_idx = int(rng_smoke.integers(0, mod.weight.numel()))

r = bfa.measure_vlm_flipped_loss_reprefill(
    model, model_inputs, gt_action,
    module=mod, flat_idx=rand_idx, bit=0,
    noise=fixed_noise, device=DEVICE, t_val=T_VAL,
)
delta = r["post_loss"] - baseline
print(f"mantissa LSB Δloss: {delta:.4g} (post_loss={r['post_loss']:.4g})")
assert r["post_loss_finite"] == 1.0, "LSB flip should not produce inf/nan"

# Verify restore: rerun baseline through the no-grad path, must match.
post_restore = loss_fn().item()
assert abs(post_restore - baseline) < 1e-5, (post_restore, baseline)
print(f"restore: OK ({post_restore})")

In [ ]:
# (d) Catastrophic exponent: bit 14 should produce inf/nan loss.
r_bad = bfa.measure_vlm_flipped_loss_reprefill(
    model, model_inputs, gt_action,
    module=mod, flat_idx=rand_idx, bit=14,
    noise=fixed_noise, device=DEVICE, t_val=T_VAL,
)
print(f"bit-14 post_loss: {r_bad['post_loss']:.4g}, finite={r_bad['post_loss_finite']}")
# Not all coords blow up to inf (depends on sign of original value); allow either.
post_restore = loss_fn().item()
assert abs(post_restore - baseline) < 1e-5, (post_restore, baseline)
print("catastrophic-bit restore: OK")

## Main sweep

16 bits × 50 modules × 5 coords × ~0.65 s/trial ≈ **45 min/scene**.
Coords come from `topk_bitflip_coords(grads, bit, k=5)` — the gradient-guided top-K. Compare against `bfa_vlm_random.ipynb` (random coords, same budget) for the gradient-vs-random ablation.

In [ ]:
BITS = list(range(16))
TOP_K_PER_MODULE = 5
# Drop dead-end modules (last layer's MLP / q_proj / o_proj have None grad —
# they don't feed the K/V cache the action expert reads). See warning from
# compute_clean_grads_vlm above.
MODULE_SUBSET = [n for n in targets.keys() if n in grads]

results = []
t_start = time.time()
for bit in BITS:
    for name in MODULE_SUBSET:
        mod = targets[name]
        w = mod.weight.data
        g = grads[name]
        flat_idx, bg_vals = bfa.topk_bitflip_coords(w, g, bit=bit, k=TOP_K_PER_MODULE)
        for i, idx in enumerate(flat_idx.tolist()):
            r = bfa.measure_vlm_flipped_loss_reprefill(
                model, model_inputs, gt_action,
                module=mod, flat_idx=idx, bit=bit,
                noise=fixed_noise, device=DEVICE, t_val=T_VAL,
            )
            r.update({
                "module": name,
                "rank": i,
                "predicted_bit_grad": float(bg_vals[i].item()),
                "baseline_loss": baseline,
                "delta_loss": r["post_loss"] - baseline,
                "ranking": "grad_topk",
            })
            results.append(r)
    print(f"bit {bit:2d} done — {len(results):5d} trials, {time.time()-t_start:.0f}s elapsed")

In [ ]:
out_dir = Path("results")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"bfa_vlm_grad_scene{SCENE_IDX}.parquet"
df = pd.DataFrame(results)
df.to_parquet(out_path)
print(f"saved {len(df)} trials to {out_path}")

In [ ]:
# Per-bit summary + finite-only top-10
print(df.groupby("bit")["delta_loss"].describe()[["count", "mean", "max"]])
df_finite = df[df["post_loss_finite"] == 1.0]
print("\nTop-10 single-bit flips by delta_loss (finite-loss only):")
print(df_finite.nlargest(10, "delta_loss")[["module", "bit", "flat_idx", "delta_loss", "predicted_bit_grad"]])

fig, ax = plt.subplots(figsize=(10, 4))
df_finite.boxplot(column="delta_loss", by="bit", ax=ax, showfliers=False)
ax.set_title("Phase 3a — Δloss by bit (finite-loss only)")
ax.set_ylabel("Δ FM loss")
plt.suptitle("")
plt.tight_layout()
plt.savefig(out_dir / f"bfa_vlm_grad_per_bit_scene{SCENE_IDX}.png", dpi=120)
plt.show()

## Phase 3a-perp: CoT perplexity ranker

Same gradient-ranking machinery, different scalar loss: NLL of the clean
model's CoT under (potentially flipped) VLM weights. The per-trial cost
is one VLM forward (~600 ms), no FMContext rebuild — slightly faster than
the FM-loss-with-reprefill loop above.


In [ ]:
# Capture the clean CoT once. Re-used across the entire perp sweep + cascade.
cot_capture = bfa.capture_clean_cot_sequences(model, model_inputs, device=DEVICE, seed=42)
S_cot = cot_capture["full_sequences"].shape[-1] - cot_capture["prefix_len"]
print(f"prefix_len: {cot_capture['prefix_len']}, S_cot: {S_cot}")
print(f"CoT (first 300 chars): {cot_capture['cot_text'][:300]}")

# Sanity: clean weights => low NLL / perplexity (model has sharp distribution
# over its own outputs). Expected NLL ~0.1-2.0, perplexity ~1.1-7.4.
with torch.no_grad():
    with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
        clean_perp_loss = bfa.cot_perplexity_loss(
            model,
            cot_capture["full_sequences"],
            cot_capture["prefix_len"],
            cot_capture["tokenized_data"],
            device=DEVICE,
        ).item()
print(f"clean CoT NLL: {clean_perp_loss:.4f}, perplexity: {np.exp(clean_perp_loss):.2f}")
